# ML Platform Research: Modeling

## 1. Introduction

This notebook evaluates machine learning models for predicting whether a visitor will make a purchase within the next seven days.

The main objective is not only to classify visitors as buyers or non-buyers, but to produce a useful probability score that can be used to rank visitors and identify groups that may be suitable for targeted incentives.

Since the data represents temporal snapshots, the modeling process preserves the chronological order of the observations to avoid training on future information.

## 2. Imports and Dataset Load

In [ ]:
import pandas as pd
import numpy as np
import tempfile
from pathlib import Path
import joblib

import sys

sys.path.append("..")

from utils import (
    evaluate_model,
    threshold_analysis,
    ranking_analysis,
    plot_roc_curves,
    plot_precision_recall_curves,
    plot_calibration,
    generate_oof_predictions,
    logit,
    calibrate_probability,
)

from sklearn.base import clone

from sklearn.compose import ColumnTransformer

from sklearn.ensemble import (
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier
)

from sklearn.inspection import permutation_importance

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

from sklearn.model_selection import (
    TimeSeriesSplit,
    RandomizedSearchCV
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier

import mlflow 
from mlflow.models import infer_signature

In [ ]:
user_aggregation = pd.read_parquet(
    "../data/processed/processed_data.parquet"
)

user_aggregation

## 3. Dataset Preparation

### 3.1 Interaction Filter

Visitors with very little recent interaction history provide limited behavioral
information for the model.

For this experiment, observations with fewer than five interactions are
excluded. The interaction count is based on views, add-to-cart events, and
transactions measured within their respective feature windows:

- Views and add-to-cart events are measured over the previous seven days,
  providing a recent representation of visitor activity.
- Transactions are measured over the previous 30 days, since purchase history
  is expected to provide a stronger behavioral signal and may remain relevant
  over a longer period.

The threshold is intended to reduce cases with very limited behavioral
information, where the available features may provide little evidence about
future purchasing behavior.

In [ ]:
interaction_columns = [
    "user_views_7d",
    "user_addtocart_7d",
    "user_transactions_30d"
]

user_aggregation = user_aggregation[
    user_aggregation[interaction_columns].sum(axis=1) >= 5
].copy()

user_aggregation

### 3.2 Temporal Boundaries

The first 30 days are removed because the feature engineering process requires historical data.

The final six days are removed because the target represents purchases occurring within the current day and the following six days.

Therefore, a complete seven-day target window is required.

In [ ]:
min_date = user_aggregation["date"].min()
max_date = user_aggregation["date"].max()

feature_cutoff = min_date + pd.Timedelta(days=30)
target_cutoff = max_date - pd.Timedelta(days=6)

model_data = user_aggregation[
    (user_aggregation["date"] >= feature_cutoff) &
    (user_aggregation["date"] <= target_cutoff)
].copy()

model_data

## 4. Features and Target

In [ ]:
features = [
    "user_views_7d",
    "user_addtocart_7d",
    "user_transactions_30d",
    "user_sessions_30d",
    "avg_session_duration_30d",
    "user_cart_items_avg_popularity_7d",
    "user_cart_top_seller_ratio_7d"
]

target = "purchase_next_7d"

In [ ]:
y = model_data[target]

y.value_counts()

In [ ]:
y.value_counts(
    normalize=True
)

## 5. Temporal Train / Validation / Test Split

The observations are divided chronologically:

- Train: model fitting and hyperparameter tuning;
- Validation: model selection and threshold/ranking analysis;
- Test: final evaluation on an unseen temporal period.

The validation set is used for model selection and development decisions.

The test set remains untouched until the final evaluation, providing an
estimate of how the selected final model generalizes to a future temporal
period.

In [ ]:
dates = (
    model_data["date"]
    .sort_values()
    .unique()
)

train_cutoff = dates[
    int(len(dates) * 0.60)
]

validation_cutoff = dates[
    int(len(dates) * 0.80)
]

In [ ]:
train_data = model_data[
    model_data["date"] < train_cutoff
].copy()

validation_data = model_data[
    (model_data["date"] >= train_cutoff) &
    (model_data["date"] < validation_cutoff)
].copy()

test_data = model_data[
    model_data["date"] >= validation_cutoff
].copy()

### 5.1 Periods

In [ ]:
print(
    "Train:",
    train_data["date"].min(),
    "→",
    train_data["date"].max()
)

print(
    "Validation:",
    validation_data["date"].min(),
    "→",
    validation_data["date"].max()
)

print(
    "Test:",
    test_data["date"].min(),
    "→",
    test_data["date"].max()
)

### 5.2 Dataset Sizes

In [ ]:
print(f"Train: {len(train_data):,}")
print(f"Validation: {len(validation_data):,}")
print(f"Test: {len(test_data):,}")

### 5.3 Positive Rate

In [ ]:
print(
    "Train positive rate:",
    train_data[target].mean()
)

print(
    "Validation positive rate:",
    validation_data[target].mean()
)

print(
    "Test positive rate:",
    test_data[target].mean()
)

### 5.4 Modeling Data

In [ ]:
X_train = train_data[features]
y_train = train_data[target]

X_validation = validation_data[features]
y_validation = validation_data[target]

X_test = test_data[features]
y_test = test_data[target]

## 6. Preprocessing

Logistic Regression benefits from standardized numerical variables.

Tree-based models do not require scaling.

### 6.1 Logistic Regression Preprocessor

In [ ]:
logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            features
        )
    ]
)

## 7. Baseline Models

Baseline models provide reference points before hyperparameter optimization.

The models are evaluated using metrics that reflect both ranking performance
and probability quality.

Average Precision is used as the primary model selection metric because the
positive class is relatively uncommon and the practical application depends
on identifying and ranking likely purchasers.

ROC-AUC is used as a secondary ranking metric.

Brier Score and calibration curves are used to evaluate the quality of the
predicted probabilities.

### 7.1 Logistic Regression Baseline

Class weighting is used to account for the imbalance between purchasers and
non-purchasers.

In [ ]:
logistic_baseline = Pipeline([
    (
        "preprocessor",
        logistic_preprocessor
    ),
    (
        "model",
        LogisticRegression(
            class_weight={
                0: 1,
                1: 12
            },
            max_iter=1000,
            random_state=42
        )
    )
])

In [ ]:
logistic_baseline.fit(
    X_train,
    y_train
)

In [ ]:
y_proba_logistic_baseline = (
    logistic_baseline
    .predict_proba(X_validation)[:, 1]
)

### 7.2 HistGradientBoosting Baseline

In [ ]:
hist_baseline = Pipeline([
    (
        "model",
        HistGradientBoostingClassifier(
            max_iter=350,
            learning_rate=0.04,
            max_depth=18,
            max_leaf_nodes=127,
            l2_regularization=0.1,
            class_weight={
                0: 1,
                1: 12
            },
            random_state=42
        )
    )
])

In [ ]:
hist_baseline.fit(
    X_train,
    y_train
)

In [ ]:
y_proba_hist_baseline = (
    hist_baseline
    .predict_proba(X_validation)[:, 1]
)

### 7.3 Baseline Comparison

In [ ]:
baseline_results = pd.DataFrame([
    {
        "model": "Baseline Logistic Regression",
        **evaluate_model(
            y_validation,
            y_proba_logistic_baseline
        )
    },
    {
        "model": "Baseline HistGradientBoosting",
        **evaluate_model(
            y_validation,
            y_proba_hist_baseline
        )
    }
])

baseline_results

## 8. Time-Aware Cross-Validation

Random cross-validation is inappropriate for this problem because observations are temporal.

Instead, TimeSeriesSplit is used during hyperparameter optimization.

In [ ]:
tscv = TimeSeriesSplit(
    n_splits=3
)

## 9. Hyperparameter Tuning

The following models are optimized using RandomizedSearchCV:

- Logistic Regression
- HistGradientBoosting
- Random Forest
- Gradient Boosting
- XGBoost

The optimization metric is Average Precision.

### 9.1 Logistic Regression

In [ ]:
logistic_model = Pipeline([
    (
        "preprocessor",
        logistic_preprocessor
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

In [ ]:
logistic_param_distributions = {
    "model__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ],

    "model__class_weight": [
        {0: 1, 1: 5},
        {0: 1, 1: 10},
        {0: 1, 1: 20},
        {0: 1, 1: 50},
        {0: 1, 1: 100},
        {0: 1, 1: 125}
    ]
}

In [ ]:
logistic_search = RandomizedSearchCV(
    estimator=logistic_model,
    param_distributions=logistic_param_distributions,
    n_iter=15,
    scoring="average_precision",
    cv=tscv,
    random_state=42,
    n_jobs=2,
    verbose=1
)

In [ ]:
logistic_search.fit(
    X_train,
    y_train
)

In [ ]:
print(
    "Best CV Average Precision:",
    logistic_search.best_score_
)

logistic_search.best_params_

### 9.2 HistGradientBoosting

In [ ]:
hist_model = Pipeline([
    (
        "model",
        HistGradientBoostingClassifier(
            random_state=42
        )
    )
])

In [ ]:
hist_param_distributions = {
    "model__learning_rate": [
        0.02,
        0.04,
        0.06,
        0.1
    ],

    "model__max_iter": [
        200,
        350,
        500
    ],

    "model__max_leaf_nodes": [
        15,
        31,
        63,
        127
    ],

    "model__max_depth": [
        5,
        10,
        15,
        None
    ],

    "model__min_samples_leaf": [
        10,
        20,
        50,
        100
    ],

    "model__l2_regularization": [
        0,
        0.1,
        1,
        10
    ],

    "model__class_weight": [
        {0: 1, 1: 20},
        {0: 1, 1: 50},
        {0: 1, 1: 100}
    ]
}

In [ ]:
hist_search = RandomizedSearchCV(
    estimator=hist_model,
    param_distributions=hist_param_distributions,
    n_iter=20,
    scoring="average_precision",
    cv=tscv,
    random_state=42,
    n_jobs=2,
    verbose=1
)

In [ ]:
hist_search.fit(
    X_train,
    y_train
)

In [ ]:
print(
    "Best CV Average Precision:",
    hist_search.best_score_
)

hist_search.best_params_

### 9.3 Random Forest

In [ ]:
rf_pipeline = Pipeline([
    (
        "model",
        RandomForestClassifier(
            random_state=42,
            n_jobs=2
        )
    )
])

In [ ]:
rf_param_distributions = {
    "model__n_estimators": [
        100,
        200,
        300
    ],
    "model__max_depth": [
        None,
        10,
        20,
        30
    ],
    "model__min_samples_split": [
        2,
        5,
        10,
        20
    ],
    "model__min_samples_leaf": [
        1,
        2,
        5,
        10
    ],
    "model__max_features": [
        "sqrt",
        "log2",
        0.5
    ],
    "model__class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}

In [ ]:
rf_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=rf_param_distributions,
    n_iter=15,
    scoring="average_precision",
    cv=tscv,
    random_state=42,
    n_jobs=2,
    verbose=1
)

In [ ]:
rf_search.fit(
    X_train,
    y_train
)

In [ ]:
print(
    "Best CV Average Precision:",
    rf_search.best_score_
)

rf_search.best_params_

### 9.4 Gradient Boosting

In [ ]:
gradient_pipeline = Pipeline([
    (
        "model",
        GradientBoostingClassifier(
            random_state=42
        )
    )
])

In [ ]:
gradient_param_distributions = {
    "model__n_estimators": [
        100,
        200,
        300
    ],
    "model__learning_rate": [
        0.02,
        0.05,
        0.1
    ],
    "model__max_depth": [
        2,
        3,
        5,
        8
    ],
    "model__min_samples_split": [
        2,
        5,
        10,
        20
    ],
    "model__min_samples_leaf": [
        1,
        5,
        10,
        20
    ],
    "model__max_features": [
        None,
        "sqrt",
        "log2"
    ]
}

In [ ]:
gradient_search = RandomizedSearchCV(
    gradient_pipeline,
    param_distributions=gradient_param_distributions,
    n_iter=15,
    scoring="average_precision",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [ ]:
gradient_search.fit(
    X_train,
    y_train
)

In [ ]:
print(
    "Best CV Average Precision:",
    gradient_search.best_score_
)

gradient_search.best_params_

### 9.5 XGBoost

In [ ]:
xg_pipeline = Pipeline([
    (
        "model",
        XGBClassifier(
            tree_method="hist",
            random_state=42,
            n_jobs=2,
            eval_metric="logloss"
        )
    )
])

In [ ]:
xg_param_distributions = {
    "model__n_estimators": [
        100,
        200,
        300,
        500
    ],
    "model__learning_rate": [
        0.02,
        0.04,
        0.06,
        0.1
    ],
    "model__max_depth": [
        3,
        5,
        7,
        9
    ],
    "model__min_child_weight": [
        1,
        3,
        5,
        10
    ],
    "model__subsample": [
        0.7,
        0.8,
        1.0
    ],
    "model__colsample_bytree": [
        0.7,
        0.8,
        1.0
    ],
    "model__reg_lambda": [
        0.1,
        1,
        5,
        10
    ]
}

In [ ]:
xg_search = RandomizedSearchCV(
    xg_pipeline,
    param_distributions=xg_param_distributions,
    n_iter=15,
    scoring="average_precision",
    cv=tscv,
    random_state=42,
    n_jobs=2,
    verbose=1
)

In [ ]:
xg_search.fit(
    X_train,
    y_train
)

In [ ]:
print(
    "Best CV Average Precision:",
    xg_search.best_score_
)

xg_search.best_params_

## 10. Model Selection

At this point, the test set has still not been used.

The models are evaluated on the validation set.

In [ ]:
models = {
    "Baseline Logistic Regression":
        logistic_baseline,

    "Baseline HistGradientBoosting":
        hist_baseline,

    "Tuned Logistic Regression":
        logistic_search.best_estimator_,

    "Tuned HistGradientBoosting":
        hist_search.best_estimator_,

    "Tuned Random Forest":
        rf_search.best_estimator_,

    "Tuned Gradient Boosting":
        gradient_search.best_estimator_,

    "Tuned XGBoost":
        xg_search.best_estimator_
}

In [ ]:
validation_predictions = {}

for name, model in models.items():

    validation_predictions[name] = (
        model
        .predict_proba(X_validation)[:, 1]
    )

In [ ]:
validation_results = pd.DataFrame([
    {
        "model": name,
        **evaluate_model(
            y_validation,
            y_proba
        )
    }

    for name, y_proba
    in validation_predictions.items()
]).sort_values(
    "pr_auc",
    ascending=False
)

validation_results

## 11. Validation Analysis

The validation set is used to understand how the selected model behaves, rather than only looking at one metric.

### 11.1 ROC Curve

In [ ]:
plot_roc_curves(
    y_validation,
    validation_predictions,
    "ROC Curve - Validation Set"
)

### 11.2 Precision-Recall Curve

In [ ]:
plot_precision_recall_curves(
    y_validation,
    validation_predictions,
    "Precision-Recall Curve - Validation Set"
)

## 12. Select Best Model

Since Average Precision is the primary model selection criterion, the model with the highest validation Average Precision is selected for further analysis.

In [ ]:
best_model_name = (
    validation_results
    .iloc[0]["model"]
)

best_model = models[
    best_model_name
]

print(
    "Selected model:",
    best_model_name
)

## 13. Threshold Analysis

Given that the project's main use case does not necessarily require binary classification, threshold analysis is used to understand the trade-off between precision and recall.

In [ ]:
thresholds = [
    0.01,
    0.05,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

In [ ]:
best_validation_proba = (
    validation_predictions[
        best_model_name
    ]
)

threshold_results = threshold_analysis(
    y_validation,
    best_validation_proba,
    thresholds
)

threshold_results

## 14. Ranking Analysis

Ranking metrics are particularly relevant because the commercial application may have limited capacity to contact or incentivize visitors.

In [ ]:
k_values = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
    0.10
]

In [ ]:
ranking_results = ranking_analysis(
    y_validation,
    best_validation_proba,
    k_values
)

ranking_results

## 15. Feature Importance

For the selected model, permutation importance is calculated using the
validation set.

Permutation importance measures the decrease in model performance when a
feature is randomly shuffled. Larger decreases indicate features that are
more useful to the model's predictions.

These results describe predictive importance and should not be interpreted as
causal effects.

In [ ]:
perm = permutation_importance(
    best_model,
    X_validation,
    y_validation,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

In [ ]:
model_importance = (
    pd.DataFrame({
        "feature": features,
        "importance": perm.importances_mean
    })
    .sort_values(
        "importance",
        ascending=False
    )
)

model_importance

## 16. ML Flow Model Validation Log

In [ ]:
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"

mlflow.set_tracking_uri(
    MLFLOW_TRACKING_URI
)

mlflow.set_experiment(
    "RetailRocket_Purchase_Prediction"
)

In [ ]:
with mlflow.start_run(run_name="validation_model_evaluation"):

    mlflow.log_param(
        "model",
        best_model_name
    )

    mlflow.log_param(
        "selection_metric",
        "average_precision"
    )

    mlflow.log_metric(
        "validation_pr_auc",
        validation_results.iloc[0]["pr_auc"]
    )

    mlflow.log_metric(
        "validation_roc_auc",
        validation_results.iloc[0]["roc_auc"]
    )

## 17. Validation Calibration

Calibration is evaluated separately from ranking performance.

A well-ranked model may still produce probabilities that are systematically too high or too low. Calibration analysis therefore evaluates whether predicted probabilities correspond reasonably well to observed frequencies.

In [ ]:
plot_calibration(
    y_validation,
    best_validation_proba,
    title=f"Calibration - {best_model_name}"
)

In [ ]:
print(
    "Validation Brier Score:",
    brier_score_loss(
        y_validation,
        best_validation_proba
    )
)

## 18. Final Model Training

We now create a development dataset consisting of Train + Validation.

Since model selection has already been completed, the validation observations can now be incorporated into the final training process.

The test set remains completely untouched.

In [ ]:
development_data = (
    pd.concat(
        [
            train_data,
            validation_data
        ],
        ignore_index=True
    )
    .sort_values("date")
    .reset_index(drop=True)
)

In [ ]:
X_development = development_data[features]
y_development = development_data[target]

print(
    "Development:",
    X_development.shape
)

print(
    "Positive rate:",
    y_development.mean()
)

## 19. Train Final Base Model

In [ ]:
final_base_model = clone(best_model)

In [ ]:
final_base_model.fit(
    X_development,
    y_development
)

## 20. Probability Calibration

The objective of calibration is to make the numerical probabilities more
representative of observed frequencies.

Because the final base model is trained on the complete development dataset,
its in-sample predictions cannot be directly used to train the calibrator.
Doing so would allow the calibrator to learn from predictions generated on
observations that the model has already seen.

Instead, out-of-fold (OOF) predictions are generated using time-aware
cross-validation.

In [ ]:
oof_indices, oof_predictions = generate_oof_predictions(
    best_model,
    X_development,
    y_development,
    tscv
)

In [ ]:
calibration_data = pd.DataFrame({
    "y": y_development.iloc[oof_indices].to_numpy(),
    "predicted_probability": oof_predictions
})

calibration_data.head()

In [ ]:
print(
    "Calibration observations:",
    len(calibration_data)
)

print(
    "Calibration positive rate:",
    calibration_data["y"].mean()
)

## 21. Fit Sigmoid Calibrator

Platt scaling is implemented using logistic regression over the log-odds of the model's predicted probability.

In [ ]:
X_calibration = logit(
    calibration_data[
        "predicted_probability"
    ].to_numpy()
).reshape(-1, 1)

y_calibration = calibration_data[
    "y"
].to_numpy()

In [ ]:
calibrator = LogisticRegression(
    max_iter=1000,
    random_state=42
)

calibrator.fit(
    X_calibration,
    y_calibration
)

In [ ]:
print(
    "Calibration coefficient:",
    calibrator.coef_[0][0]
)

print(
    "Calibration intercept:",
    calibrator.intercept_[0]
)

## 22. Final Test Predictions

This is the first point where the test set is used after all modeling decisions have been made.

In [ ]:
y_proba_test_raw = (
    final_base_model
    .predict_proba(X_test)[:, 1]
)

In [ ]:
y_proba_test_calibrated = calibrate_probability(
    y_proba_test_raw,
    calibrator
)

## 23. Final Evaluation

We evaluate both versions for comparison.

In [ ]:
final_results = pd.DataFrame([
    {
        "model": f"{best_model_name} - Raw",
        "roc_auc": roc_auc_score(
            y_test,
            y_proba_test_raw
        ),
        "pr_auc": average_precision_score(
            y_test,
            y_proba_test_raw
        ),
        "brier_score": brier_score_loss(
            y_test,
            y_proba_test_raw
        )
    },

    {
        "model": f"{best_model_name} - Calibrated",
        "roc_auc": roc_auc_score(
            y_test,
            y_proba_test_calibrated
        ),
        "pr_auc": average_precision_score(
            y_test,
            y_proba_test_calibrated
        ),
        "brier_score": brier_score_loss(
            y_test,
            y_proba_test_calibrated
        )
    }
])

final_results

## 24. Final ROC Curve

Because sigmoid calibration is a monotonic transformation, the ranking should remain unchanged.

In [ ]:
plot_roc_curves(
    y_test,
    {
        f"{best_model_name} - Raw":
            y_proba_test_raw,

        f"{best_model_name} - Calibrated":
            y_proba_test_calibrated
    },
    "ROC Curve - Final Test Set"
)

## 25. Final Precision-Recall Curve

In [ ]:
plot_precision_recall_curves(
    y_test,
    {
        f"{best_model_name} - Raw":
            y_proba_test_raw,

        f"{best_model_name} - Calibrated":
            y_proba_test_calibrated
    },
    "Precision-Recall Curve - Final Test Set"
)

## 26. Final Ranking Performance

The calibrated probabilities are used for the final ranking analysis.

In [ ]:
final_ranking_results = ranking_analysis(
    y_test,
    y_proba_test_calibrated,
    k_values
)

final_ranking_results

## 27. Final Calibration

In [ ]:
plot_calibration(
    y_test,
    y_proba_test_calibrated,
    title=f"Final Calibration - {best_model_name}"
)

In [ ]:
print(
    "Final Brier Score:",
    brier_score_loss(
        y_test,
        y_proba_test_calibrated
    )
)

## 28. Final Predictions

The calibrated probability is used as the final score.

In [ ]:
model_predictions = test_data[
    [
        "visitorid",
        "date"
    ]
].copy()

model_predictions["purchase_probability"] = (
    y_proba_test_calibrated
)

In [ ]:
model_predictions = (
    model_predictions
    .sort_values(
        "purchase_probability",
        ascending=False
    )
)

model_predictions.head(20)

## 29. ML Flow Final Evaluation Log

In [ ]:
with mlflow.start_run(
    run_name="final_evaluation"
):

    mlflow.log_param(
        "model",
        best_model_name
    )

    mlflow.log_metric(
        "test_raw_roc_auc",
        roc_auc_score(
            y_test,
            y_proba_test_raw
        )
    )

    mlflow.log_metric(
        "test_raw_pr_auc",
        average_precision_score(
            y_test,
            y_proba_test_raw
        )
    )

    mlflow.log_metric(
        "test_raw_brier",
        brier_score_loss(
            y_test,
            y_proba_test_raw
        )
    )

    mlflow.log_metric(
        "test_calibrated_roc_auc",
        roc_auc_score(
            y_test,
            y_proba_test_calibrated
        )
    )

    mlflow.log_metric(
        "test_calibrated_pr_auc",
        average_precision_score(
            y_test,
            y_proba_test_calibrated
        )
    )

    mlflow.log_metric(
        "test_calibrated_brier",
        brier_score_loss(
            y_test,
            y_proba_test_calibrated
        )
    )

## 30. Build Production Model

The final prediction process consists of two sequential components:

1. the trained prediction model generates the raw purchase probability;
2. the Platt scaling calibrator transforms the raw probability into the final calibrated probability.

For production use, these components are wrapped into a single MLflow PyFunc model.

This allows the inference pipeline to consume one model interface without
depending on the internal implementation of the prediction and calibration
steps.

### 30.1 Building

In [ ]:
tmp_dir = Path(
    tempfile.mkdtemp()
)

prediction_model_path = (
    tmp_dir / "prediction_model.joblib"
)

calibrator_path = (
    tmp_dir / "calibrator.joblib"
)

joblib.dump(
    final_base_model,
    prediction_model_path,
)

joblib.dump(
    calibrator,
    calibrator_path,
)

### 30.2 Model Signature

In [ ]:
signature_input = X_test.head(5).copy()

raw_probability = (
    final_base_model
    .predict_proba(signature_input)[:, 1]
)

signature_output = calibrate_probability(
    raw_probability,
    calibrator,
)

signature = infer_signature(
    signature_input,
    signature_output,
)

input_example = signature_input.head(1)

## 31. Save Final Model

The final production model combines the trained prediction model and the probability calibrator into a single MLflow PyFunc model.

The individual components are also logged as artifacts for research transparency and reproducibility, while `purchase_model` represents the production inference interface.

### 31.1 Log and Register Model

In [ ]:
with mlflow.start_run(
    run_name="final_model"
):

    mlflow.log_param(
        "model",
        best_model_name,
    )

    mlflow.log_param(
        "calibration",
        "Platt Scaling",
    )

    mlflow.sklearn.log_model(
        final_base_model,
        "prediction_model",
        serialization_format="cloudpickle",
    )

    mlflow.sklearn.log_model(
        calibrator,
        "probability_calibrator",
        serialization_format="cloudpickle",
    )

    model_info = mlflow.pyfunc.log_model(
        name="purchase_model",
        python_model=(
            "../model_definition/"
            "purchase_prediction_model.py"
        ),
        artifacts={
            "prediction_model": str(
                prediction_model_path
            ),
            "calibrator": str(
                calibrator_path
            ),
        },
        signature=signature,
        input_example=input_example,
        registered_model_name="RetailRocketPurchaseModel",
    )

In [ ]:
loaded_model = mlflow.pyfunc.load_model(
    model_info.model_uri
)

### 31.2 Production Model Equivalence Check

In [ ]:
X_production_check = signature_input.copy()

reference_predictions = calibrate_probability(
    final_base_model.predict_proba(
        X_production_check
    )[:, 1],
    calibrator,
)

production_predictions = loaded_model.predict(
    X_production_check
)

In [ ]:
max_difference = np.max(
    np.abs(
        reference_predictions
        - production_predictions
    )
)

print(
    "Maximum prediction difference:",
    max_difference
)

print(
    "Equivalent:",
    np.allclose(
        reference_predictions,
        production_predictions,
        rtol=1e-10,
        atol=1e-12,
    )
)

### 31.3 Set Production Alias

In [ ]:
client = mlflow.MlflowClient()

client.set_registered_model_alias(
    name="RetailRocketPurchaseModel",
    alias="production",
    version=model_info.registered_model_version,
)

print(
    f"Production alias → version "
    f"{model_info.registered_model_version}"
)

## 32. Conclusion

This notebook developed and evaluated a machine learning model to predict
whether a visitor will make a purchase within the next seven days, as part of
the broader objective of building a machine learning pipeline for e-commerce
behavior analysis.

The modeling process used chronological train, validation, and test periods to
reduce the risk of temporal leakage. Multiple classification algorithms were
compared and optimized using time-aware cross-validation, with Average
Precision used as the primary model selection metric due to the imbalance of
the target.

The selected XGBoost model demonstrated strong performance on the unseen test
period. The ranking results, particularly Precision@K, Recall@K, and Lift@K,
indicate that the model can effectively prioritize visitors with a higher
likelihood of purchase.

Probability calibration was also applied using out-of-fold predictions and
Platt scaling. This provides a more interpretable probability score while
preserving the ranking behavior of the model.

The final output is a calibrated purchase probability for each visitor. These
probabilities can be used to support downstream segmentation and
prioritization, such as identifying groups of visitors with different levels
of purchase propensity.

However, the model estimates purchase propensity rather than the causal effect
of a commercial intervention. The available data does not contain treatment
and control information, so the model cannot determine whether offering an
incentive would actually increase purchases. Additionally, the absence of
transaction monetary values prevents the model from directly optimizing
revenue or customer value.

Overall, the modeling results provide a predictive component that can be
integrated into the broader machine learning platform. Future improvements
within the modeling stage could include additional behavioral features,
monitoring for temporal drift, and the use of causal or uplift modeling when
appropriate treatment and control data becomes available.